In [41]:
import pandas as pd 
import numpy as np
import tensorflow as tf  

In [42]:
df=pd.read_csv('adult.csv')
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 6.2 MB


In [44]:
x=df.drop('income', axis=1)
y=df['income']

In [45]:
x

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States


In [46]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
label=LabelEncoder()
y=label.fit_transform(y)

In [47]:
y

array([0, 0, 0, ..., 1, 0, 0], shape=(32561,))

In [48]:
cat_features=x.select_dtypes(include='str').columns 
num_features=x.select_dtypes(exclude='str').columns 


In [49]:
num_features

Index(['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss',
       'hours.per.week'],
      dtype='str')

In [50]:
num_transform=StandardScaler()
cat_transform=OneHotEncoder(handle_unknown='ignore')

In [51]:
from sklearn.compose import ColumnTransformer

scaler=ColumnTransformer(
    transformers=[
        ('OnehotEncoder', cat_transform, cat_features),
        ('StandardScaler', num_transform, num_features)
    ]
)

In [52]:
x=scaler.fit_transform(x)

In [55]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 455854 stored elements and shape (32561, 108)>

In [56]:
from sklearn.model_selection import train_test_split 
x_train,x_test, y_train,y_test=train_test_split(x, y, test_size=0.2, random_state=42)

In [57]:
x_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 364672 stored elements and shape (26048, 108)>

In [58]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard


In [60]:
model=Sequential(
    [
        Dense(128, activation='relu', input_shape=(x_train.shape[1], )),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ]
)

d:\PROJECTS\DEEP LEARNING PROJECT\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [61]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [63]:
earlystopping=EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [64]:
history=model.fit(
    x_train, y_train,
    validation_data=[x_test, y_test],
    epochs=25,
    callbacks=[earlystopping]
)

Epoch 1/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8487 - loss: 0.3281 - val_accuracy: 0.8508 - val_loss: 0.3160
Epoch 2/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8590 - loss: 0.3060 - val_accuracy: 0.8546 - val_loss: 0.3102
Epoch 3/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8610 - loss: 0.2998 - val_accuracy: 0.8514 - val_loss: 0.3133
Epoch 4/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8648 - loss: 0.2953 - val_accuracy: 0.8549 - val_loss: 0.3102
Epoch 5/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8668 - loss: 0.2905 - val_accuracy: 0.8511 - val_loss: 0.3134
Epoch 6/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8703 - loss: 0.2840 - val_accuracy: 0.8580 - val_loss: 0.3128
Epoch 7/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8730 - loss: 0.2774 - val_accuracy: 0.8544 - val_loss: 0.3154
Epoch 8/25
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8733 - loss: 0.2727 - val_accuracy: 0.

In [65]:
prediction=model.predict(x_test)

204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [66]:
prediction

array([[0.11727548],
       [0.001143  ],
       [0.39682782],
       ...,
       [0.7123393 ],
       [0.00100285],
       [0.23924401]], shape=(6513, 1), dtype=float32)